# 05 — Analysis

Load the trained checkpoint, evaluate per-state prediction accuracy,
and compare the model's learned transition probabilities to the true
Gambler's Ruin matrix.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# -- Papermill parameters -----------------------------------------------------
DATA_DIR = "projects/markov-transformer/experiments/markov-chain-learning/data"
EMBEDDING_VARIANT = "full"  # used to find the right checkpoint file

In [ ]:
# -- Post-papermill coercions -------------------------------------------------
DATA_DIR = Path(DATA_DIR)
print("Data dir:", DATA_DIR.resolve())

## Load model & data

In [ ]:
# Model definition (self-contained copy)
class AttentionOnlyBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=0.0,
            batch_first=True,
        )

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        out, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        return x + out


class MarkovTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,
        d_model=32,
        nhead=4,
        num_attention_layers=2,
        dim_feedforward=64,
        max_len=64,
        use_token_embedding=True,
        use_pos_embedding=True,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.use_token_embedding = use_token_embedding
        self.use_pos_embedding = use_pos_embedding

        if use_token_embedding:
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            self.token_projection = None
        else:
            self.token_embedding = None
            self.token_projection = nn.Linear(vocab_size, d_model, bias=False)

        if use_pos_embedding:
            self.pos_embedding = nn.Embedding(max_len, d_model)
        else:
            self.pos_embedding = None

        self.attention_blocks = nn.ModuleList(
            [
                AttentionOnlyBlock(d_model=d_model, nhead=nhead)
                for _ in range(num_attention_layers)
            ]
        )

        self.mlp_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Linear(dim_feedforward, d_model),
        )
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, L = x.shape

        if self.use_token_embedding:
            h = self.token_embedding(x)
        else:
            x_one_hot = F.one_hot(x, num_classes=self.vocab_size).float()
            h = self.token_projection(x_one_hot)

        if self.use_pos_embedding:
            positions = torch.arange(L, device=x.device)
            h = h + self.pos_embedding(positions)

        mask = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device), diagonal=1
        )
        for block in self.attention_blocks:
            h = block(h, attn_mask=mask)

        h = h + self.mlp(self.mlp_norm(h))
        return self.head(h)

In [ ]:
checkpoint_path = DATA_DIR / "checkpoint.pt"
if not checkpoint_path.exists():
    candidates = sorted(DATA_DIR.glob("checkpoint_*.pt"))
    if not candidates:
        raise FileNotFoundError("No checkpoint found in data directory.")
    checkpoint_path = max(candidates, key=lambda p: p.stat().st_mtime)

ckpt = torch.load(checkpoint_path, weights_only=True)
cfg = ckpt["config"]
training_meta = ckpt.get("training", {})

model = MarkovTransformer(**cfg)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(
    f"Loaded {checkpoint_path.name} from epoch {ckpt['epoch']} "
    f"(val loss {ckpt['best_val_loss']:.4f})"
)

data = torch.load(DATA_DIR / "sequences.pt", weights_only=True)
Ts = data["Ts"]
pis = data["pis"]
seqs = data["sequences"]
chain_to_sample = data["chain_to_sample"]
data_cfg = data.get("config", {})

VOCAB = int(cfg["vocab_size"])
PAD_ID = int(data_cfg.get("pad_id", -1))

if Ts.shape[0] == 1:
    T_reference = Ts[0]
    pi_reference = pis[0]
else:
    chain_weights = torch.bincount(chain_to_sample, minlength=Ts.shape[0]).float()
    chain_weights = chain_weights / chain_weights.sum()
    T_reference = (chain_weights[:, None, None] * Ts).sum(dim=0)
    pi_reference = (chain_weights[:, None] * pis).sum(dim=0)

print(f"DGP regime: {data_cfg.get('dgp_regime', 'unknown')}")
print(f"Embedding variant: {training_meta.get('embedding_variant', 'unknown')}")
print(f"Reference T shape: {tuple(T_reference.shape)}")

## Generalization loss (analytic)

For a single-chain DGP, this computes the exact expected per-token NLL under the stationary Markov process.
For multi-chain DGPs, the same formula is a one-step stationary mixture surrogate (not an exact full-history value).

In [ ]:
all_states = torch.arange(VOCAB).unsqueeze(1)
with torch.no_grad():
    logits_single = model(all_states)[:, 0, :]  # (V, V)
    log_probs = torch.log_softmax(logits_single, dim=-1)

expected_nll = -(pi_reference[:, None] * T_reference * log_probs).sum().item()
optimal_nll = (
    -(pi_reference[:, None] * T_reference * torch.log(T_reference.clamp_min(1e-12)))
    .sum()
    .item()
)
excess_nll = expected_nll - optimal_nll

print(f"Expected per-token NLL: {expected_nll:.6f}")
print(f"Bayes-optimal NLL (reference chain): {optimal_nll:.6f}")
print(f"Excess NLL over Bayes: {excess_nll:.6f}")
if Ts.shape[0] > 1:
    print("Note: multi-chain result is a one-step stationary mixture surrogate.")

## Per-state next-token accuracy

In [ ]:
# For each position t in each sequence, check whether argmax(logits[t]) == seq[t+1].
# Break out accuracy by the *current* state (seq[t]) to get a per-state view.
correct = torch.zeros(VOCAB)
totals = torch.zeros(VOCAB)

BATCH = 512
with torch.no_grad():
    for start in range(0, len(seqs), BATCH):
        batch = seqs[start : start + BATCH]  # (B, L)
        x_in = batch[:, :-1]  # (B, L-1)
        x_tgt = batch[:, 1:]  # (B, L-1)
        logits = model(x_in)  # (B, L-1, V)
        preds = logits.argmax(dim=-1)  # (B, L-1)

        valid = x_tgt != PAD_ID
        for state in range(VOCAB):
            mask = (x_in == state) & valid
            totals[state] += mask.sum()
            correct[state] += (preds[mask] == x_tgt[mask]).sum()

acc = (correct / totals.clamp(min=1)).numpy()
print("Per-state accuracy:")
for s in range(VOCAB):
    bar = "#" * int(acc[s] * 40)
    print(f"  State {s}: {acc[s]:.1%}  {bar}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(VOCAB), acc)
ax.set_xticks(range(VOCAB))
ax.set_xlabel("Current state")
ax.set_ylabel("Next-token accuracy")
ax.set_title("Per-state next-token prediction accuracy")
ax.set_ylim(0, 1)
ax.axhline(
    1.0 / VOCAB, color="gray", linestyle="--", linewidth=0.8, label=f"p=1/{VOCAB}"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Learned vs true transition matrix

Feed each state as a single-token context; the model's softmax output at
position 0 approximates T[state, :] (the next-state distribution given
a one-step history).

In [ ]:
# Single-token context: shape (VOCAB, 1)
all_states = torch.arange(VOCAB).unsqueeze(1)
with torch.no_grad():
    logits_single = model(all_states)  # (VOCAB, 1, VOCAB)
    T_learned = logits_single[:, 0, :].softmax(dim=-1)  # (VOCAB, VOCAB)

print("Learned T (single-step context):")
print(T_learned.numpy().round(3))
print("\nReference T:")
print(T_reference.numpy().round(3))
print("\nMax absolute error:", (T_learned - T_reference).abs().max().item())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

kw = dict(
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    linewidths=0.3,
    xticklabels=[f"→{j}" for j in range(VOCAB)],
    yticklabels=[str(i) for i in range(VOCAB)],
)

sns.heatmap(T_reference.numpy(), ax=axes[0], **kw)
axes[0].set_title("Reference T")

sns.heatmap(T_learned.detach().numpy(), ax=axes[1], **kw)
axes[1].set_title("Learned T (single-step)")

diff = (T_learned - T_reference).detach().numpy()
sns.heatmap(
    diff,
    ax=axes[2],
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    xticklabels=[f"→{j}" for j in range(VOCAB)],
    yticklabels=[str(i) for i in range(VOCAB)],
)
axes[2].set_title("Difference  (learned − reference)")

for ax in axes:
    ax.set_xlabel("Next state")
    ax.set_ylabel("Current state")

plt.suptitle("Transformer transition probabilities vs reference", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## Multi-step context: does more history help?

The Markov property implies that *only* the current state matters.
Feed longer prefixes ending in each state and check whether the full-context
predictions converge to the true T row.

In [ ]:
from markov_chain import sample_sequences

# For each state i, collect sequences that are at state i at their last position,
# then compare the model's predicted distribution to the chosen reference row.
CONTEXT_LEN = 8
T_probe = Ts[0]
pi_probe = pis[0]

test_seqs_np = sample_sequences(
    T_probe.numpy(), n=5_000, seq_len=CONTEXT_LEN + 1, prior=pi_probe.numpy(), rng=99
)
test_seqs = torch.from_numpy(test_seqs_np).long()

T_ctx = torch.zeros(VOCAB, VOCAB)
counts_ctx = torch.zeros(VOCAB)

with torch.no_grad():
    logits_ctx = model(test_seqs[:, :-1])  # (N, CONTEXT_LEN, V)
    probs_ctx = logits_ctx[:, -1, :].softmax(dim=-1)  # (N, V)

for state in range(VOCAB):
    mask = test_seqs[:, -2] == state  # sequences where pos-before-last == state
    if mask.sum() == 0:
        continue
    T_ctx[state] = probs_ctx[mask].mean(dim=0)
    counts_ctx[state] = mask.sum()

print(f"Avg predicted T (context len {CONTEXT_LEN}):")
print(T_ctx.numpy().round(3))
print("Max abs error (vs probe chain 0):", (T_ctx - T_probe).abs().max().item())